## Дерево решений без масштабирования

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate

# Загрузка данных
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
X = train.drop(columns=["Id", "Survived"]).copy()
y = train["Survived"]
X_test = test.drop(columns="Id").copy()
for data in [X, X_test]:
    data["around_whealth"] = (data["Parch"] + data["SibSp"]) * data["Fare"]


## Подготовка признаков

In [2]:
# Преобразования внутри фолдов
numeric_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare", "around_whealth"]
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Пропуск")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_columns),
    ("categorical", categorical_transformer, ["Sex", "Embarked"]),
])
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("tree", DecisionTreeClassifier(random_state=42)),
])


## Подбор параметров

In [3]:
# Пятикратная кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    "tree__max_depth": [3, 5, 7, None],
    "tree__min_samples_leaf": [1, 3, 5, 10],
    "tree__criterion": ["gini", "entropy"],
}
search = GridSearchCV(
    pipeline, param_grid, scoring={"f1": "f1", "roc_auc": "roc_auc"},
    refit="f1", cv=cv, n_jobs=-1, return_train_score=True,
)
search.fit(X, y)
print("Лучшие параметры:", search.best_params_)
print("F1 при подборе:", round(search.best_score_, 4))
cv_results = pd.DataFrame(search.cv_results_)
display(cv_results[["params", "mean_train_f1", "mean_test_f1", "std_test_f1", "mean_test_roc_auc"]]
        .sort_values("mean_test_f1", ascending=False).head(10))


Лучшие параметры: {'tree__criterion': 'entropy', 'tree__max_depth': None, 'tree__min_samples_leaf': 10}
F1 при подборе: 0.7725


,params,mean_train_f1,mean_test_f1,std_test_f1,mean_test_roc_auc
31,"{'tree__criterion': 'entropy', 'tree__max_dept...",0.825484,0.772540,0.044453,0.842321
27,"{'tree__criterion': 'entropy', 'tree__max_dept...",0.825142,0.771266,0.040523,0.845909
15,"{'tree__criterion': 'gini', 'tree__max_depth':...",0.829763,0.769218,0.036054,0.843804
11,"{'tree__criterion': 'gini', 'tree__max_depth':...",0.831046,0.764795,0.036335,0.841214
9,"{'tree__criterion': 'gini', 'tree__max_depth':...",0.858346,0.761025,0.041521,0.828528
10,"{'tree__criterion': 'gini', 'tree__max_depth':...",0.846979,0.758276,0.040841,0.836169
26,"{'tree__criterion': 'entropy', 'tree__max_dept...",0.833998,0.756665,0.027812,0.836228
24,"{'tree__criterion': 'entropy', 'tree__max_dept...",0.857720,0.749975,0.034085,0.818152
14,"{'tree__criterion': 'gini', 'tree__max_depth':...",0.857634,0.748767,0.040596,0.833638
7,"{'tree__criterion': 'gini', 'tree__max_depth':...",0.804241,0.747913,0.044451,0.847749


## Независимая оценка подбора

In [4]:
# Вложенная кросс-валидация
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
nested_scores = cross_validate(
    search, X, y, cv=outer_cv, scoring={"f1": "f1", "roc_auc": "roc_auc"},
    n_jobs=1,
)
fold_results = pd.DataFrame({
    "Фолд": range(1, 6),
    "F1": nested_scores["test_f1"],
    "ROC AUC": nested_scores["test_roc_auc"],
})
display(fold_results)
print("F1:", round(fold_results["F1"].mean(), 4), "±", round(fold_results["F1"].std(ddof=0), 4))
print("ROC AUC:", round(fold_results["ROC AUC"].mean(), 4), "±", round(fold_results["ROC AUC"].std(ddof=0), 4))


,Фолд,F1,ROC AUC
0,1,0.761194,0.798485
1,2,0.721311,0.869452
2,3,0.739496,0.833556
3,4,0.733813,0.814572
4,5,0.720000,0.799827


F1: 0.7352 ± 0.015
ROC AUC: 0.8232 ± 0.0264


## Предсказания теста

In [5]:
# Посылка отдельного дерева
best_tree = search.best_estimator_
submission = pd.DataFrame({"Id": test["Id"], "Survived": best_tree.predict(X_test)})
sample = pd.read_csv("data/sample_submission.csv")
assert submission.columns.tolist() == sample.columns.tolist()
assert submission["Id"].equals(sample["Id"])
assert submission["Survived"].isin([0, 1]).all()
assert not submission.isna().any().any()
submission.to_csv("submission_decision_tree_unscaled_cv.csv", index=False)
print("Сохранён submission_decision_tree_unscaled_cv.csv")


Сохранён submission_decision_tree_unscaled_cv.csv
